In [1]:
import spider_agent_workbench.paths
import spider_agent_workbench.loaders as loaders
import spider_agent_workbench.agent as agent_workbench

In [2]:
from spider_agent_workbench.eval.sql_result_scorer import score_query

In [3]:
# Get the available example from train_spider dataset and group them by db
spider_train_examples = list(loaders.iter_examples('train'))
filtered_examples = loaders.filter_available(spider_train_examples)
grouped_examples = loaders.group_by_db(filtered_examples)

available_dbs = grouped_examples.keys()
print (len(available_dbs))

140


In [4]:
# Select database with less than 20 questions so that we can test in v1
target_db_dict = {db:spider_examples for db, spider_examples in grouped_examples.items() if len(spider_examples) < 15}
print (len(target_db_dict))
for db_id, spider_examples in target_db_dict.items():
    print (f"db_id: {db_id}, num_questions: {len(spider_examples)}")

3
db_id: soccer_1, num_questions: 14
db_id: company_1, num_questions: 7
db_id: local_govt_mdm, num_questions: 14


In [5]:
# From the selected 'soccer_1' database with 14 questions, list all available questions and their correct answers (golden queries)
select_spider_db = 'soccer_1'
selected_spider_examples = target_db_dict[select_spider_db]
for example in selected_spider_examples:
    print (example)

SpiderExample(db_id='soccer_1', question='List all country and league names.', gold_sql='SELECT T1.name ,  T2.name FROM Country AS T1 JOIN League AS T2 ON T1.id  =  T2.country_id')
SpiderExample(db_id='soccer_1', question='How many leagues are there in England?', gold_sql='SELECT count(*) FROM Country AS T1 JOIN League AS T2 ON T1.id  =  T2.country_id WHERE T1.name  =  "England"')
SpiderExample(db_id='soccer_1', question='What is the average weight of all players?', gold_sql='SELECT avg(weight) FROM Player')
SpiderExample(db_id='soccer_1', question='What is the maximum and minimum height of all players?', gold_sql='SELECT max(weight) ,  min(weight) FROM Player')
SpiderExample(db_id='soccer_1', question='List all player names who have an overall rating higher than the average.', gold_sql='SELECT DISTINCT T1.player_name FROM Player AS T1 JOIN Player_Attributes AS T2 ON T1.player_api_id = T2.player_api_id WHERE T2.overall_rating  >  ( SELECT avg(overall_rating) FROM Player_Attributes )')


In [6]:
# create an agent that we can use everytime (no need to create a fresh agent again)
spider_agent = agent_workbench.build_agent()

In [7]:
# Give a question to the agent and see the answer
question = selected_spider_examples[0].question
print (f"Question: {question}")
answer = agent_workbench.answer_question(select_spider_db, question, spider_agent)
print (f"Answer: {answer}")
ai_query_score = score_query(select_spider_db, answer.sql, selected_spider_examples[0].gold_sql)
print (ai_query_score)


Question: List all country and league names.
Answer: AgentAnswer(db_id='soccer_1', question='List all country and league names.', sql='\nSELECT \n    c.name AS Country_Name, \n    l.name AS League_Name\nFROM Country c\nJOIN League l ON c.id = l.country_id\n', turns=5, hit_turn_limit=False)
ScoreResult(score=1, status='match', detail=None)
